# Lesson 21: Stereo Matching

Two cameras viewing the same scene from slightly different positions see the same 3D points shifted by different amounts depending on depth &mdash; nearby points shift more, distant points shift less. **Stereo matching** finds these shifts (the **disparity**) at every pixel, which triangulation then converts into depth. The matching step itself is close cousin to two things we've already built: block matching (SSD, the sum-of-squared-differences idea) and 1D optical flow (Lesson 20), just restricted to a single row instead of two dimensions.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Rectified stereo: why the search is 1D

For a pair of cameras that are side-by-side, pointed the same direction, with parallel image planes (a **rectified** stereo pair &mdash; real camera rigs are calibrated and warped to approximate this), a fundamental fact of epipolar geometry applies: the corresponding point for any pixel in the left image lies **on the same row** in the right image. This collapses the search for a match from a 2D image search down to a 1D scan along one row, and the horizontal offset between the two matching positions is the **disparity** $d$.

Disparity relates to depth by

$$Z = \frac{f \cdot B}{d}$$

where $f$ is the focal length and $B$ is the baseline (distance between the two camera centers). Depth is **inversely proportional to disparity**: nearby objects have large disparity, distant objects have small disparity, and an object infinitely far away has zero disparity.

## A synthetic stereo pair with known ground truth

We build a left image of pure random texture (so every patch is locally distinctive &mdash; no aperture-problem ambiguity) and construct the right image by shifting each pixel left by its true disparity, which we set to three different constant values for three depth "planes": a background and two nearer rectangles.

In [ ]:
rng = np.random.default_rng(0)
h, w = 150, 200
left = rng.integers(0, 255, (h, w)).astype(np.uint8)

true_disparity = np.full((h, w), 5, dtype=np.int32)     # background
true_disparity[30:100, 50:150] = 15                       # a nearer rectangle
true_disparity[60:90, 80:120] = 25                        # an even nearer rectangle

right = np.full((h, w), -1, dtype=np.int32)
for y in range(h):
    for x in range(w):
        xr = x - true_disparity[y, x]
        if 0 <= xr < w:
            right[y, xr] = left[y, x]
occluded = right == -1   # positions no left pixel maps to: disocclusions, filled with fresh noise
right[occluded] = rng.integers(0, 255, occluded.sum())
right = right.astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, im, title in zip(axes, [left, right, true_disparity],
                          ['Left image', 'Right image', 'Ground-truth disparity']):
    ax.imshow(im, cmap='gray' if title != 'Ground-truth disparity' else 'viridis')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Block matching along the scanline

For each pixel in the left image, we slide a small block along the *same row* of the right image over a range of candidate disparities and keep the disparity with the lowest sum-of-squared-differences &mdash; the same SSD idea used for template matching, just restricted to a 1D search.

In [ ]:
def block_match_stereo(left, right, block_size=7, max_disp=30):
    h, w = left.shape
    half = block_size // 2
    left_f, right_f = left.astype(np.float64), right.astype(np.float64)
    disparity_map = np.zeros((h, w), dtype=np.float64)

    for y in range(half, h - half):
        for x in range(half, w - half):
            left_patch = left_f[y - half:y + half + 1, x - half:x + half + 1]
            best_d, best_cost = 0, np.inf
            for d in range(max_disp + 1):
                xr = x - d
                if xr - half < 0:
                    break
                right_patch = right_f[y - half:y + half + 1, xr - half:xr + half + 1]
                cost = ((left_patch - right_patch)**2).sum()
                if cost < best_cost:
                    best_cost, best_d = cost, d
            disparity_map[y, x] = best_d
    return disparity_map

# a small crop, since the pure-Python pixel loop is slow
crop = np.s_[0:40, 0:60]
manual_disp = block_match_stereo(left[crop], right[crop], block_size=7, max_disp=30)

interior = np.s_[10:30, 10:30]
error = np.abs(manual_disp[interior] - true_disparity[crop][interior])
print(f'mean absolute disparity error (interior of crop): {error.mean():.3f} pixels')

With pure, unambiguous random texture and a matching direction that's correct by construction, the block matcher recovers the true disparity exactly. Real images are far less forgiving &mdash; repetitive texture, flat regions, and occlusions all introduce ambiguity, which is exactly what makes stereo matching a genuinely hard problem in practice.

## The same thing, at full resolution, with OpenCV

`cv2.StereoBM` implements this same block-matching idea (with some efficiency and post-processing refinements) fast enough to run on the whole image.

In [ ]:
stereo_bm = cv2.StereoBM_create(numDisparities=32, blockSize=9)
disparity_bm = stereo_bm.compute(left, right).astype(np.float32) / 16.0  # fixed-point output, divide to get pixels

print('recovered disparity vs. ground truth, by region:')
print(f'  background (true=5):  {disparity_bm[100:110, 150:160].mean():.2f}')
print(f'  mid layer (true=15):  {disparity_bm[40:50, 60:70].mean():.2f}')
print(f'  near layer (true=25): {disparity_bm[70:80, 90:110].mean():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(true_disparity, cmap='viridis', vmin=0, vmax=30)
axes[0].set_title('Ground truth')
im = axes[1].imshow(np.where(disparity_bm >= 0, disparity_bm, np.nan), cmap='viridis', vmin=0, vmax=30)
axes[1].set_title('cv2.StereoBM output\n(gray = invalid/low-confidence)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

`StereoBM` marks a pixel invalid (returns $-1$) wherever it isn't confident in a unique match, e.g. too close to the image border to search the full disparity range.

## Block size: a detail vs. noise tradeoff

A larger matching block averages over more pixels, making the match more robust to noise but blurring across depth discontinuities (mixing pixels from two different true depths into one "averaged" disparity estimate near object edges). A smaller block preserves sharp depth boundaries but is more easily fooled by noise or repetitive texture.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, block_size in zip(axes, [5, 15, 31]):
    stereo = cv2.StereoBM_create(numDisparities=32, blockSize=block_size)
    d = stereo.compute(left, right).astype(np.float32) / 16.0
    ax.imshow(np.where(d >= 0, d, np.nan), cmap='viridis', vmin=0, vmax=30)
    ax.set_title(f'blockSize={block_size}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## From disparity to a depth map

Given a focal length and baseline (in whatever consistent units), $Z = fB/d$ converts a disparity map directly into a metric depth map.

In [ ]:
focal_length_px = 500.0
baseline_m = 0.1

valid = disparity_bm > 0
depth_m = np.full_like(disparity_bm, np.nan)
depth_m[valid] = focal_length_px * baseline_m / disparity_bm[valid]

plt.imshow(depth_m, cmap='viridis_r')
plt.colorbar(label='estimated depth (m)')
plt.title('Depth map (nearer = brighter)')
plt.axis('off')
plt.show()

### Exercise

1. Replace the random-texture `left` image with a flat gray region for the background (keeping the two rectangles textured). Run `cv2.StereoBM` again and describe what happens to the disparity estimate over the flat area &mdash; this is the aperture problem from Lesson 20, now in a stereo-matching context.
2. Try `cv2.StereoSGBM_create` (semi-global block matching, which enforces smoothness across neighboring disparities rather than matching each pixel completely independently) in place of `StereoBM`. Compare the amount of speckle noise in flat/background regions between the two methods.
3. Increase `baseline_m` in the depth conversion. How does the estimated depth map change, and why would a wider-baseline stereo rig give more precise depth estimates for distant objects (at the cost of a larger minimum-distance blind spot, due to disparity ranges and occlusion, near the cameras)?